# Retirement Tax Optimizer — Spouse A & Spouse B (package edition)

**Goal:** maximize after-tax terminal net worth at the planning horizon, subject to never running out of money, by jointly choosing:

1. **Pre-retirement** allocation between Traditional 401(k) and Roth 401(k) for each spouse.
2. **In-retirement** withdrawal sequencing across taxable / pretax / Roth buckets.
3. **Roth-conversion** size during the retirement-to-RMD gap years.

All simulation logic lives in the `tax_optimizer/` Python package. This notebook is a thin orchestration layer that:

- Imports the engine and configures a scenario.
- Runs the four-strategy comparison + tornado sensitivity (deterministic).
- Demonstrates the v2 capabilities:
  - **Monte Carlo** sequence-of-returns risk (`§8`).
  - **Widow's-penalty / single-filer mortality** stress test (`§9`).
  - **Tax-regime change** (TCJA sunset) stress test (`§10`).
  - **Smile-shaped retirement spending** + lump events (`§11`).
  - **Asset location** (per-account equity / bond mix) (`§12`).

To use your own scenario, edit cell `§1` and re-run.


## §1. Scenario inputs

In [ ]:
from __future__ import annotations
from dataclasses import replace

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tax_optimizer import (
    Config,
    Inputs,
    StartingBalances,
    CurrentIncome,
    CurrentContrib,
    PensionInputs,
    SocialSecurity,
    SpendingProfile,
    SpendingPhase,
    LumpEvent,
    LongTermCareShock,
    AssetLocation,
    AssetMix,
    DeterministicModel,
    LognormalModel,
    BootstrapModel,
    HistoricalSequenceModel,
    CMA_PRESETS,
    lognormal_from_cma,
    Mortality,
    TCJA_EXTENDED,
    PRE_TCJA_2017,
    SUNSET_2026,
    simulate,
    simulate_paths,
    summarize,
    optimize_s3,
    BRACKET_CHOICES,
    tornado_sensitivity,
    render_actions,
    render_takeaways,
    StrategyResult,
)
from tax_optimizer.tax import federal_tax, irmaa_annual_surcharge


In [ ]:
# --- DataFrame display helper -----------------------------------------------
# `fmt_money(df)` renders `df` with currency / percent / integer formatting
# per known column. Returns a `Styler` (so `df` itself is unchanged) and
# avoids the scientific notation pandas falls back to for large floats.
_DOLLAR_COLS = {
    # year-by-year simulator output
    'wages', 'pension', 'ssn', 'rmd', 'rmd_a', 'rmd_b',
    'roth_conversion', 'roth_conversion_a', 'roth_conversion_b',
    'pretax_withdrawal', 'pretax_withdrawal_a', 'pretax_withdrawal_b',
    'roth_withdrawal', 'taxable_withdrawal',
    'agi', 'taxable_income', 'federal_tax', 'irmaa', 'spending_need',
    'pretax_balance', 'pretax_a_balance', 'pretax_b_balance',
    'roth_balance', 'taxable_balance', 'hsa_balance', 'pension_balance',
    # `summarize()` keys
    'lifetime_tax_npv', 'lifetime_irmaa_npv', 'terminal_after_tax', 'min_balance',
    # `MonteCarloResult.summary()` keys
    'terminal_p5', 'terminal_p25', 'terminal_p50', 'terminal_p75', 'terminal_p95',
    'cvar_terminal_p10', 'cvar_terminal_p20',
    'lifetime_tax_p50', 'lifetime_irmaa_p50',
    # `tornado_sensitivity` deltas
    'delta_low', 'delta_high', 'swing',
}
_PCT_COLS = {'marginal', 'peak_marginal', 'prob_success', 'equity_return', 'bond_return'}
_INT_COLS = {
    'year', 'spouse_a_age', 'spouse_b_age', 'irmaa_tier',
    'years_irmaa', 'peak_irmaa_tier', 'n_paths',
}


def fmt_money(df):
    """Return a Styler that formats `df` for display.

    Dollar columns render as `$1,234,567`, percent columns as `12.3%`,
    integer-coded columns as `1,234`. Unknown columns are left alone.
    `median_ruin_year_offset == -1` (the "no ruin" sentinel) renders as `—`.
    """
    fmt = {}
    for col in df.columns:
        c = str(col)
        if c in _DOLLAR_COLS:
            fmt[col] = '${:,.0f}'
        elif c in _PCT_COLS:
            fmt[col] = '{:.1%}'
        elif c == 'median_ruin_year_offset':
            fmt[col] = lambda v: '—' if v < 0 else f'{v:,.0f}'
        elif c in _INT_COLS:
            fmt[col] = '{:,.0f}'
    return df.style.format(fmt)


In [ ]:
# -- Scenario: typical middle-class dual-income MFJ couple turning 50 in 2026 --
# Defaults reflect median-ish U.S. data for households at peak earning years:
#   - Combined gross ~$145k (Pew middle-class band for a 3-person household).
#   - Retirement balances near the median for age 50 (Vanguard "How America Saves").
#   - No private pension, modest taxable brokerage and HSA.
inputs = Inputs(
    # --- Ages (Spouse A is the simulation anchor for cfg.horizon_age) ----
    spouse_a_age_start=50,           # current age in cfg.start_year (defaults to 2026)
    spouse_b_age_start=50,
    spouse_a_retire_age=65,          # last year of W-2 wages and 401(k) contributions
    spouse_b_retire_age=65,
    # --- 401(k) deferral policy (decision variables for the S3 optimizer) -
    spouse_a_total_contrib_pct=0.08, # fraction of GROSS salary deferred to 401(k)
    spouse_b_total_contrib_pct=0.06,
    spouse_a_roth_401k_pct=0.0,      # share of the deferral routed to Roth (rest = Traditional)
    spouse_b_roth_401k_pct=0.0,
    # --- Today's account balances (in current dollars) -------------------
    starting=StartingBalances(
        spouse_a_pretax_401k=225_000.0,
        spouse_b_pretax_401k=150_000.0,
        spouse_a_roth_ira=40_000.0,     # pooled with B's Roth IRA in state.roth
        spouse_b_roth_ira=0.0,          # documentation-only split; simulator pools both
        spouse_a_pretax_ira=0.0,        # pooled with A's 401(k) in state.spouse_a_pretax (RMDs use A's age)
        spouse_b_pretax_ira=35_000.0,   # pooled with B's 401(k) in state.spouse_b_pretax (RMDs use B's age)
        pension_balance=0.0,            # cash-balance-style pension; 0 if none
        hsa=18_000.0,
        taxable_brokerage=80_000.0,     # cost basis tracked dynamically via cfg.cap_gains_basis_fraction
    ),
    # --- Annual gross income (modeled only during working years) ---------
    income=CurrentIncome(
        spouse_a_gross=95_000.0,       # W-2 wages BEFORE any 401(k) deferral
        spouse_b_gross=70_000.0,
        spouse_a_bonus=5_000.0,        # taxed as ordinary wages; not subject to 401(k) deferral here
        interest=500.0,               # 1099-INT; ordinary rates
        capital_gains=1_000.0,         # realized LTCG; preferential rate
        dividends=2_000.0,              # qualified dividends; preferential rate
    ),
    # --- HSA family contribution ------------------------------------------
    # Pre-tax annual HSA target. The simulator caps this at the IRS
    # family-coverage limit (+$1,000 catch-up if either spouse is 55+),
    # subtracts it from Box-1 wages, and stops contributing once both
    # spouses hit Medicare eligibility at 65. Set to 0 to disable.
    contrib=CurrentContrib(
        hsa_family=8_550.0,             # 2026 IRS family HSA cap
    ),
    # --- Defined-benefit / cash-balance pension --------------------------
    # `monthly_at_nrd` = monthly annuity expected at Normal Retirement Date
    # (= inputs.pension.start_age). Leave balance / annuity at 0 if there's no pension.
    pension=PensionInputs(balance_today=0.0, monthly_at_nrd=0.0, start_age=65),
    # --- Social Security -------------------------------------------------
    # Monthly benefit paid once each spouse reaches inputs.ss.start_age.
    # There is no FRA actuarial adjustment in the model, so enter the
    # amount you expect to receive AT the claim age. Survivor inherits the
    # larger of the two monthly amounts when one spouse dies.
    ss=SocialSecurity(monthly_spouse_a=2_700.0, monthly_spouse_b=2_200.0, start_age=70),
    # --- Retirement spending (today's dollars; inflated each year) -------
    annual_expenses=85_000.0,
)

# -- Default Config: deterministic 6% growth, MFJ, horizon age 90 ------------
# `Config` now only carries simulation-level knobs (macro assumptions, age-
# gated policy events, regime / market / spending blocks). The household
# "about me" data — ages, retire ages, contribution rates — lives on
# `Inputs` above so the same `Config` can be reused across households.
cfg = Config(
    annual_expenses_today=inputs.annual_expenses,
    nominal_growth_rate=0.06,    # equity & bond return when market is deterministic
    inflation=0.025,             # CPI; inflates expenses, brackets, SS, IRMAA tiers
    wage_growth=0.030,           # nominal raises applied until each spouse's retire age
    horizon_age=90,              # last simulated age for Spouse A; B's age tracks independently
    # rmd_start_age=75,          # SECURE 2.0 default; change only to model future legislation
    # SS claim age and pension NRD now live on inputs.ss.start_age /
    # inputs.pension.start_age (above), not on Config.
)

print('--- Starting balances ---')
for k, v in inputs.starting.__dict__.items():
    print(f'  {k:30s} ${v:>14,.0f}')
print(f'\nAnnual expenses:    ${inputs.annual_expenses:,.0f}')
print(f'Horizon:           {inputs.spouse_a_age_start} -> {cfg.horizon_age}')
print(f'Tax regime:        {cfg.tax_regime.name}')

In [ ]:
if 1 != 1:
      # --- Alternate scenario: load (cfg_family1, inputs_family1) from JSON --------
      # This block is parallel to the inline `cfg` / `inputs` defined above; it
      # does *not* overwrite them. To switch the notebook over to the scenario
      # file's values, re-bind `cfg, inputs = cfg_family1, inputs_family1` here
      # (or wherever you'd like the switch to take effect) and re-run the
      # downstream cells.
      from pathlib import Path

      from tax_optimizer import apply_scenario, load_scenario_file

      _SCENARIO_PATH = Path('scenarios/example01.json')
      _scenario = load_scenario_file(_SCENARIO_PATH)
      cfg, inputs = apply_scenario(Config(), Inputs(), _scenario)

      print(f'Loaded scenario: {_SCENARIO_PATH}')
      print(f'  Spouse A age / retire / Roth-401(k) %: '
            f'{inputs.spouse_a_age_start} / '
            f'{inputs.spouse_a_retire_age} / '
            f'{inputs.spouse_a_roth_401k_pct:.0%}')
      print(f'  Spouse B age / retire / Roth-401(k) %: '
            f'{inputs.spouse_b_age_start} / '
            f'{inputs.spouse_b_retire_age} / '
            f'{inputs.spouse_b_roth_401k_pct:.0%}')
      print(f'  Horizon age:           {cfg.horizon_age}')
      print(f'  Tax regime:            {cfg.tax_regime.name}')
      print(f'  Annual expenses today: ${cfg.annual_expenses_today:,.0f}')

      print('--- Starting balances ---')
      for k, v in inputs.starting.__dict__.items():
            print(f'  {k:30s} ${v:>14,.0f}')
      print(f'\nAnnual expenses:    ${inputs.annual_expenses:,.0f}')
      print(f'Horizon:           {inputs.spouse_a_age_start} -> {cfg.horizon_age}')
      print(f'Tax regime:        {cfg.tax_regime.name}')


## §2. Sanity check — first-year federal tax

Verify the package's federal-tax engine produces sensible numbers
against a hand calculation for the scenario's first working year.


In [ ]:
# First-year wages, pre-tax 401(k) deferrals + HSA reduce wages_box1.
# Top-level Inputs fields are the source of truth for deferral pcts.
a_pretax = inputs.income.spouse_a_gross * inputs.spouse_a_total_contrib_pct * (1 - inputs.spouse_a_roth_401k_pct)
b_pretax = inputs.income.spouse_b_gross * inputs.spouse_b_total_contrib_pct * (1 - inputs.spouse_b_roth_401k_pct)
wages_box1 = (
    inputs.income.spouse_a_gross + inputs.income.spouse_a_bonus
    + inputs.income.spouse_b_gross
    - a_pretax - b_pretax - inputs.contrib.hsa_family
)

ftax = federal_tax(
    regime=cfg.tax_regime,
    filing_status='mfj',
    wages=wages_box1,
    interest=inputs.income.interest,
    qualified_div=inputs.income.dividends,
    ltcg=inputs.income.capital_gains,
)
print(f'Year 1 AGI:           ${ftax["agi"]:,.0f}')
print(f'Year 1 taxable inc:   ${ftax["taxable_income"]:,.0f}')
print(f'Year 1 federal tax:   ${ftax["tax"]:,.0f}')
print(f'Year 1 marginal rate: {ftax["marginal"]:.0%}')


## §3. Strategy comparison: S0 / S1 / S2 / S3

Four candidate strategies, all evaluated deterministically (single
6%-growth path). The optimizer (`S3`) uses `differential_evolution`
because the IRMAA cliffs make the objective non-smooth.


In [ ]:
s0_cfg, s0_inputs = cfg, inputs
s1_cfg, s1_inputs = cfg, replace(inputs, spouse_a_roth_401k_pct=1.0, spouse_b_roth_401k_pct=1.0)
s2_cfg, s2_inputs = replace(cfg, roth_conversion_target_bracket=0.22), inputs
s3_cfg, s3_inputs, x_opt = optimize_s3(cfg, inputs, objective='terminal')
print(f'S3 optimizer chose:  Spouse A Roth %={s3_inputs.spouse_a_roth_401k_pct:.2f}  '
      f'Spouse B Roth %={s3_inputs.spouse_b_roth_401k_pct:.2f}  '
      f'Conv bracket={s3_cfg.roth_conversion_target_bracket:.0%}')

# Each strategy carries its own (cfg, inputs) pair because the optimizer
# may flip Roth-401(k) splits (which live on Inputs) and bracket-fill
# targets (which live on Config). `StrategyResult` keeps these named so
# downstream consumers don't depend on tuple ordering.
results: dict[str, StrategyResult] = {}
for name, c, inp in [
    ('S0_baseline',        s0_cfg, s0_inputs),
    ('S1_all_roth_401k',   s1_cfg, s1_inputs),
    ('S2_bracket_fill_22', s2_cfg, s2_inputs),
    ('S3_optimized',       s3_cfg, s3_inputs),
]:
    df = simulate(c, inp)
    results[name] = StrategyResult(cfg=c, inputs=inp, df=df, summary=summarize(df))

summary_df = pd.DataFrame({n: r.summary for n, r in results.items()}).T
fmt_money(summary_df[['lifetime_tax_npv', 'lifetime_irmaa_npv', 'terminal_after_tax', 'peak_marginal', 'years_irmaa']])


## §4. Visualizations: balances, taxes, contributions

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8), sharex=True)
for name, r in results.items():
    df = r.df
    liquid = df['pretax_balance'] + df['roth_balance'] + df['taxable_balance']
    axes[0,0].plot(df['spouse_a_age'], liquid, label=name)
    axes[0,1].plot(df['spouse_a_age'], df['federal_tax'], label=name)
    axes[1,0].plot(df['spouse_a_age'], df['pretax_balance'], label=name, linestyle='-')
    axes[1,0].plot(df['spouse_a_age'], df['roth_balance'], label=f'{name} (Roth)', linestyle='--')
    axes[1,1].plot(df['spouse_a_age'], df['agi'], label=name)
axes[0,0].set_title('Liquid balance (pretax + Roth + taxable)')
axes[0,1].set_title('Federal tax / year')
axes[1,0].set_title('Pretax (solid) vs Roth (dashed)')
axes[1,1].set_title('AGI / year')
for ax in axes.flat:
    ax.legend(fontsize=7)
    ax.set_xlabel('Spouse A age')
plt.tight_layout()
plt.show()


## §5. Year-by-year detail of the winning strategy

Each row is one simulation year. Key columns:
- `wages` / `pension` / `ssn` — gross income items.
- `rmd_a` / `rmd_b` — required minimum distributions (per IRS Uniform Lifetime).
- `roth_conversion` — pretax → Roth conversion in gap years.
- `pretax_withdrawal` / `taxable_withdrawal` — strategy-driven retirement draws.
- `agi` / `federal_tax` — IRC §1 computation for the active filing status.
- `irmaa` / `irmaa_tier` — Medicare premium surcharge (Tier 0 = no surcharge).
- `marginal` — top federal ordinary bracket reached.
- `*_balance` columns — end-of-year liquid balances per bucket.
- `equity_return` / `bond_return` — that year's draws from the active `MarketModel`
  (constant for `DeterministicModel`, stochastic otherwise).


In [ ]:
winner_name = max(results, key=lambda n: results[n].summary['terminal_after_tax'])
winner_df = results[winner_name].df
print(f'Winning strategy: {winner_name}')

show_cols = [
    'year','spouse_a_age','spouse_b_age','filing_status','wages','pension','ssn',
    'rmd_a','rmd_b','roth_conversion','pretax_withdrawal','taxable_withdrawal',
    'agi','federal_tax','irmaa','irmaa_tier','marginal',
    'pretax_a_balance','pretax_b_balance','roth_balance','taxable_balance',
]
fmt = {col: '${:>12,.0f}' for col in show_cols if col not in
       ('year','spouse_a_age','spouse_b_age','filing_status','irmaa_tier')}
fmt['marginal'] = '{:.0%}'
fmt['irmaa_tier'] = '{:>2.0f}'
winner_df[show_cols].iloc[:].style.format(fmt)


## §6. Tornado sensitivity

In [ ]:
sens_df, base_terminal = tornado_sensitivity(cfg, inputs)
print(f'Base terminal NW: ${base_terminal:,.0f}\n')

fig, ax = plt.subplots(figsize=(8, 6))
y = np.arange(len(sens_df))
ax.barh(y, sens_df['delta_high'].values / 1e3, alpha=0.7, label='high - base')
ax.barh(y, sens_df['delta_low'].values / 1e3, alpha=0.7, label='low - base')
ax.set_yticks(y); ax.set_yticklabels(sens_df['param'].str.replace('_', ' '))
ax.invert_yaxis(); ax.axvline(0, color='black', lw=0.8)
ax.set_xlabel('Change in terminal NW ($K)')
ax.set_title('Tornado: parameter sensitivity'); ax.legend()
plt.tight_layout(); plt.show()
fmt_money(sens_df.head(8))


## §7. Recommended actions + takeaways

In [ ]:
print(render_takeaways(results, cfg))
print()
print(render_actions(results, sens_df, cfg, base_terminal))


## §8. Monte Carlo — sequence-of-returns risk

The deterministic simulator above assumes a flat 6%/yr return every year.
That hides "sequence-of-returns risk": a market crash early in retirement
hurts a withdrawing portfolio far more than the same crash late in life.

Below we re-run the same `cfg` with a `LognormalModel` (independent yearly
draws, μ=7%, σ=18% for equities) and a `BootstrapModel` (block-bootstrap
from 1928–2023 historical S&P + 10y-Treasury data). The summary reports:

- `prob_success` — fraction of paths whose liquid balance never falls
  below that year's spending need.
- `terminal_p5/p50/p95` — 5th / median / 95th percentile of terminal
  after-tax NW.
- `cvar_terminal_p10` — average terminal NW across the worst-10% paths
  (the metric a risk-averse retiree should actually optimize on).

Use `simulate_paths(..., n_paths=2000)` for tighter tails; the 200-path
demo below runs in a few seconds.


In [ ]:
# Four return models, same household, same seed:
#   * lognormal:        IID bivariate-normal (now with equity-bond correlation)
#   * lognormal+CMA:    Vanguard 2025 forward-looking assumptions (lower μ, higher ρ)
#   * bootstrap:        block-bootstrap of 1928-2023 history
#   * historical_seq:   single contiguous 30y slice — sequence-of-returns risk
#                       in its purest form (Bengen / FIRECalc methodology)
cfg_lognormal = replace(cfg, market=LognormalModel(
    equity_mu=0.07, equity_sigma=0.18,
    bond_mu=0.04, bond_sigma=0.06,
    equity_bond_corr=0.10,        # NEW: long-run US has been mildly positive
))
cfg_vanguard  = replace(cfg, market=lognormal_from_cma('vanguard_2025'))
cfg_bootstrap = replace(cfg, market=BootstrapModel(block_size=5))
cfg_histseq   = replace(cfg, market=HistoricalSequenceModel())

mc_lognormal = simulate_paths(cfg_lognormal, inputs, n_paths=2000, seed=42)
mc_vanguard  = simulate_paths(cfg_vanguard,  inputs, n_paths=2000, seed=42)
mc_bootstrap = simulate_paths(cfg_bootstrap, inputs, n_paths=2000, seed=42)
mc_histseq   = simulate_paths(cfg_histseq,   inputs, n_paths=2000, seed=42)

mc_summary = pd.DataFrame({
    'lognormal (default)':         mc_lognormal.summary(),
    'lognormal + vanguard_2025':   mc_vanguard.summary(),
    'bootstrap (1928-2023)':       mc_bootstrap.summary(),
    'historical_sequence':         mc_histseq.summary(),
}).T
fmt_money(mc_summary)


In [ ]:
# Fan chart of liquid balance across all 200 paths.
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, mc, title in [(axes[0], mc_lognormal, 'Lognormal'),
                      (axes[1], mc_bootstrap, 'Bootstrap (1928-2023)')]:
    ages = mc.paths[0]['spouse_a_age'].to_numpy()
    arr = np.stack([
        (df['pretax_balance'] + df['roth_balance'] + df['taxable_balance']).to_numpy()
        for df in mc.paths
    ]) / 1e6
    p5, p25, p50, p75, p95 = [np.percentile(arr, p, axis=0) for p in (5,25,50,75,95)]
    ax.fill_between(ages, p5, p95, alpha=0.15, label='5-95%')
    ax.fill_between(ages, p25, p75, alpha=0.30, label='25-75%')
    ax.plot(ages, p50, lw=2, label='median')
    ax.set_title(f'{title}: liquid balance fan ({mc.n_paths} paths)')
    ax.set_xlabel('Spouse A age'); ax.set_ylabel('Liquid ($M)'); ax.legend()
plt.tight_layout(); plt.show()


## §9. Widow's-penalty stress test

Realistic plans must stress the case where one spouse dies and the
survivor switches from MFJ to single-filer status. The single-filer
penalty:

- Brackets compress (the 22% bracket caps at ~$103k for single vs $207k MFJ).
- Standard deduction halves.
- IRMAA thresholds for Medicare premiums roughly halve.
- Survivor receives only the *larger* of the two SS checks, not both.
- Pension annuity scales by the joint-and-survivor election (default 50%).

Below we model Spouse A dying at year 25 (age 75) and re-run S3.


In [ ]:
cfg_widow = replace(s3_cfg, mortality=Mortality(year_of_death_a=25,
                                               pension_survivor_pct=0.5))

df_baseline = results['S3_optimized'].df
df_widow = simulate(cfg_widow, inputs)

cmp = pd.DataFrame({
    'baseline (both alive)': summarize(df_baseline),
    'widow (A dies year 25)': summarize(df_widow),
}).T
fmt_money(cmp)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
liq_b = df_baseline['pretax_balance'] + df_baseline['roth_balance'] + df_baseline['taxable_balance']
liq_w = df_widow['pretax_balance'] + df_widow['roth_balance'] + df_widow['taxable_balance']
axes[0].plot(df_baseline['spouse_a_age'], liq_b/1e6, label='baseline')
axes[0].plot(df_widow['spouse_a_age'], liq_w/1e6, label='widow @y25')
axes[0].axvline(75, color='red', ls=':', alpha=0.6, label='death year')
axes[0].set_title('Liquid balance ($M)'); axes[0].set_xlabel('Spouse A age'); axes[0].legend()
axes[1].plot(df_baseline['spouse_a_age'], df_baseline['marginal']*100, label='baseline')
axes[1].plot(df_widow['spouse_a_age'], df_widow['marginal']*100, label='widow @y25')
axes[1].axvline(75, color='red', ls=':', alpha=0.6)
axes[1].set_title('Federal marginal bracket (%)'); axes[1].set_xlabel('Spouse A age'); axes[1].legend()
plt.tight_layout(); plt.show()


## §10. Tax-regime change — TCJA sunset stress test

The current `TCJA_EXTENDED` regime assumes Congress extends TCJA past
its scheduled 2025 expiration. If TCJA actually expires, brackets revert
to roughly the pre-TCJA structure with widths inflation-adjusted forward
(~1.30× the 2017 nominals; standard deduction roughly halves).

We model this with `regime_change_year_offset` + `regime_change_target`,
swapping to `SUNSET_2026` at year 5 (i.e. starting in 2031).


In [ ]:
cfg_sunset = replace(s3_cfg, regime_change_year_offset=5,
                                    regime_change_target=SUNSET_2026)
df_sunset = simulate(cfg_sunset, s3_inputs)

# Re-optimize S3 *under* the sunset assumption to see how the optimizer
# responds — typically conversions become more attractive in years before
# the regime change.
cfg_sunset_base = replace(cfg, regime_change_year_offset=5,
                                regime_change_target=SUNSET_2026)
s3_sunset_cfg, s3_sunset_inputs, _ = optimize_s3(cfg_sunset_base, inputs, objective='terminal')
df_sunset_opt = simulate(s3_sunset_cfg, s3_sunset_inputs)

cmp = pd.DataFrame({
    'baseline (TCJA forever)':       summarize(df_baseline),
    'sunset @y5 (S3 unchanged)':     summarize(df_sunset),
    'sunset @y5 (re-optimized S3)':  summarize(df_sunset_opt),
}).T
fmt_money(cmp)


In [ ]:
print(f'Original S3 conv target:        {s3_cfg.roth_conversion_target_bracket:.0%}')
print(f'Re-optimized under sunset:      {s3_sunset_cfg.roth_conversion_target_bracket:.0%}')
print(f'Original S3 Roth-401(k) (A/B):  {s3_inputs.spouse_a_roth_401k_pct:.0%} / {s3_inputs.spouse_b_roth_401k_pct:.0%}')
print(f'Re-optimized Roth-401(k) (A/B): {s3_sunset_inputs.spouse_a_roth_401k_pct:.0%} / {s3_sunset_inputs.spouse_b_roth_401k_pct:.0%}')


## §11. Smile-shaped retirement spending + lump events

Real spending follows the Blanchett "retirement smile":

- 100% of base in working years
- 115% of base in the "go-go" years (65–74; travel, hobbies)
- 95% of base in the "slow-go" years (75–84)
- 100% in the "no-go" years (85+) plus an LTC shock in the last 3
  years ($80k/yr today's dollars).

Plus we'll model a $50k wedding gift in year 15 and a $60k new car in year 22.


In [ ]:
smile = SpendingProfile.retirement_smile(
    base_spending=inputs.annual_expenses, inflation=0.025,
    ltc_years=3, ltc_annual_today=80_000.0,
)
smile.lump_events = [
    LumpEvent(year_offset=15, amount_today=50_000.0, label='kid wedding'),
    LumpEvent(year_offset=22, amount_today=60_000.0, label='replace car'),
]

cfg_smile = replace(s3_cfg, spending=smile)
df_smile = simulate(cfg_smile, inputs)

cmp = pd.DataFrame({
    'flat $85k (baseline S3)':            summarize(df_baseline),
    'smile + LTC + 2 lump events':        summarize(df_smile),
}).T
fmt_money(cmp)


In [ ]:
# Show the actual spending need year-by-year so the smile shape is visible.
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(df_baseline['spouse_a_age'], df_baseline['spending_need']/1e3, label='flat (baseline)')
ax.plot(df_smile['spouse_a_age'],    df_smile['spending_need']/1e3,    label='smile + events')
ax.set_title('Annual spending need ($K, nominal)')
ax.set_xlabel('Spouse A age'); ax.set_ylabel('$K'); ax.legend()
plt.tight_layout(); plt.show()


## §12. Asset location — bonds in pretax, equities in Roth

By default the package's `AssetLocation()` puts:

- 40% equity / 60% bond in the **pretax** sleeve  — bonds get sheltered
  from the ordinary-income tax line they would otherwise feed.
- 100% equity in the **Roth** sleeve              — maximize tax-free compounding.
- 80% equity / 20% bond in the **taxable** sleeve.
- 80% equity / 20% bond in the **HSA** sleeve.

Pair that with a `LognormalModel` so equity and bond returns differ
year-to-year, and the asset-location effect becomes visible.


In [ ]:
cfg_asset_uniform = replace(s3_cfg, market=LognormalModel(),
                                  asset_location=AssetLocation.uniform(equity_pct=0.6))
cfg_asset_located = replace(s3_cfg, market=LognormalModel(),
                                  asset_location=AssetLocation())   # bonds-in-pretax default

mc_uniform = simulate_paths(cfg_asset_uniform, inputs, n_paths=2000, seed=7)
mc_located = simulate_paths(cfg_asset_located, inputs, n_paths=2000, seed=7)

cmp = pd.DataFrame({
    'uniform 60/40 everywhere':           mc_uniform.summary(),
    'asset-located (bonds in pretax)':    mc_located.summary(),
}).T
fmt_money(cmp)


## §13. Action report — optimized scenario

A decision-ready summary that pulls together the winning strategy from §3,
the year-by-year detail from §5, the tornado from §6, and (if run) the
Monte Carlo risk picture from §8 into one tailored document.

The cell below regenerates the report from whatever variables are in
scope, so re-running with a different scenario or a different optimizer
objective will produce an updated plan automatically.


In [ ]:
from IPython.display import Markdown, display
from datetime import datetime
from pathlib import Path

# Use the canonical `build_action_report` from the package so this notebook
# and the `python -m tax_optimizer` CLI render byte-identical reports.
# Future updates to the report only need to land in one place
# (`tax_optimizer/report.py`).
from tax_optimizer import build_action_report

# Pull Monte Carlo from the lognormal run if it's been computed; otherwise
# the risk section is omitted gracefully.
try:
    _mc_for_report = mc_lognormal
except NameError:
    _mc_for_report = None

_report_md = build_action_report(
    cfg=cfg,
    inputs=inputs,
    results=results,
    sens_df=sens_df,
    base_terminal=base_terminal,
    mc=_mc_for_report,
)
display(Markdown(_report_md))

# Persist two copies: a stable `action_report.md` at the project root for
# quick reference, and a timestamped archive in `reports/` so re-runs
# don't clobber prior plans.
_now = datetime.now()
_generated_at = _now.strftime('%Y-%m-%d %H:%M:%S')
_filename_stamp = _now.strftime('%Y-%m-%d_%H%M%S')

_reports_dir = Path('reports')
_reports_dir.mkdir(exist_ok=True)

_body = (
    f'<!-- Generated {_generated_at} from tax_optimizer_standalone.ipynb -->\n\n'
    + _report_md + '\n'
)

_latest_path  = Path('action_report.md').resolve()
_archive_path = (_reports_dir / f'action_report_{_filename_stamp}.md').resolve()
_latest_path.write_text(_body, encoding='utf-8')
_archive_path.write_text(_body, encoding='utf-8')

print('Saved Markdown report to:')
print(f'  latest:  {_latest_path}')
print(f'  archive: {_archive_path}')


## §14. Market-model deep dive — CMAs, CAPE, and historical replay

Four ways to feed forward returns into the simulator, in order of
increasing realism for *forecasting* (vs. backward-looking history):

| Model | When to use it | Key dial |
|---|---|---|
| `LognormalModel(...)` | quick parametric Monte Carlo | `equity_mu`, `equity_sigma`, **`equity_bond_corr`** |
| `lognormal_from_cma("vanguard_2025")` | use a published forward-looking CMA | preset name + per-knob overrides |
| `LognormalModel(..., cape_today=33.0)` | valuation-aware Shiller scaling | `cape_today` (vs. `cape_long_run`) |
| `BootstrapModel(block_size=5)` | preserve short-run regimes (vol clusters) | `block_size` |
| `HistoricalSequenceModel()` | "what would *real history* have done?" | (none — exact 1928-2023 slices) |

The cells below let you compare them side-by-side. If terminal-NW
percentiles diverge meaningfully across models, you've learned which
return-assumption your plan is most sensitive to.

In [ ]:
# --- 14a. CMA preset cheat sheet ---------------------------------------------
# Long-run forward-looking expected returns / vols published by major asset
# managers. Numbers below reflect roughly 2024-2025 vintages and should be
# refreshed annually as new CMAs come out. Forward-looking CMAs typically
# bake in current valuation levels, so they sit BELOW the 1928-2023 mean.
cma_df = pd.DataFrame(CMA_PRESETS).T[
    ['equity_mu', 'equity_sigma', 'bond_mu', 'bond_sigma', 'equity_bond_corr']
]
cma_df.columns = ['μ_eq', 'σ_eq', 'μ_b', 'σ_b', 'ρ(eq,b)']
cma_df.style.format({
    'μ_eq': '{:.1%}', 'σ_eq': '{:.1%}',
    'μ_b':  '{:.1%}', 'σ_b':  '{:.1%}',
    'ρ(eq,b)': '{:+.2f}',
}).set_caption('Capital Markets Assumption presets (2024-2025 vintage)')

In [ ]:
# --- 14b. CAPE conditioning ---------------------------------------------------
# Shiller's robust empirical finding: starting CAPE explains ~40% of
# subsequent 10y equity-return variance. Setting `cape_today` scales
# `equity_mu` by `cape_long_run / cape_today`. Volatility is left
# untouched (vol doesn't scale with valuation the same way).
_cape_rows = []
for _cape in [None, 16.5, 22.0, 28.0, 33.0, 40.0]:
    _m = LognormalModel(equity_mu=0.07, cape_today=_cape, cape_long_run=16.5)
    _label = f'CAPE = {_cape}' if _cape is not None else 'CAPE off (default)'
    _cape_rows.append({
        'scenario': _label,
        'effective μ_eq': _m.effective_equity_mu(),
    })
_cape_df = pd.DataFrame(_cape_rows).set_index('scenario')
_cape_df.style.format({'effective μ_eq': '{:.2%}'}).set_caption(
    "CAPE-conditioning impact on equity_mu (raw μ = 7%, long-run CAPE = 16.5)"
)

In [ ]:
# --- 14c. 5-way market-model bake-off (single deterministic path each) -------
# Same household, same RNG seed, five different return assumptions. The spread
# in terminal NW is the honest "your-plan-depends-on-this" picture.
from tax_optimizer.metrics import terminal_after_tax_nw

_models = {
    'lognormal default (μ=7%)':            LognormalModel(),
    'lognormal vanguard_2025':             lognormal_from_cma('vanguard_2025'),
    'lognormal CAPE=33 (today-ish)':       LognormalModel(equity_mu=0.07, cape_today=33.0),
    'bootstrap (1928-2023, block=5)':      BootstrapModel(block_size=5),
    'historical_sequence (real slice)':    HistoricalSequenceModel(),
}
_rows = []
for _name, _model in _models.items():
    _cfg = replace(cfg, market=_model)
    _df  = simulate(_cfg, inputs, rng=np.random.default_rng(42))
    _rows.append({
        'model': _name,
        'terminal_after_tax': terminal_after_tax_nw(
            _df, heir_marginal_rate=cfg.heir_marginal_rate
        ),
        'lifetime_tax': float(_df['federal_tax'].sum()
                              + _df['state_tax'].sum()
                              + _df['irmaa'].sum()),
    })
_bakeoff = pd.DataFrame(_rows).set_index('model')
fmt_money(_bakeoff)

## §15. Tier B feature cheat sheet — state tax, IRA, mega-backdoor, HSA-after-65, bequest

The recent Tier B work added five new modeling capabilities. None of them
require notebook changes to *enable* (they're all opt-in via `Config` or
`Inputs` knobs), but the cell below lists every new field so you can copy
into your own scenario. Full reference: [`docs/scenario_guide.md`](docs/scenario_guide.md).

In [ ]:
# --- 15. Tier B knobs reference ----------------------------------------------
# Every new Tier B field, where it lives, and what it does.
from tax_optimizer.tax.state import STATELESS, CA, NY, IL, MA

_tier_b = pd.DataFrame([
    # State income tax
    ('Config.state_regime',                   'STATELESS / CA / NY / IL / MA',
     'state income tax — progressive brackets, retirement exclusions, HSA add-backs'),
    ('Config.state_regime_change_year_offset', 'int | None',
     'year offset when household moves states'),
    ('Config.state_regime_change_target',     'StateTaxRegime | None',
     'destination regime after the move'),
    # Bequest tax
    ('Config.heir_marginal_rate',             '0.0 - 0.40',
     'discount on terminal pretax & HSA balances when projecting to heirs'),
    # IRA contributions
    ('Inputs.spouse_*_traditional_ira_contrib', 'dollars/yr',
     'deductible Traditional IRA target (subject to MAGI/cap)'),
    ('Inputs.spouse_*_roth_ira_contrib',      'dollars/yr',
     'direct Roth IRA target (auto-phases out by MAGI)'),
    ('Inputs.spouse_*_backdoor_roth',         'bool',
     'enable backdoor Roth (pro-rata aware) when over the income limit'),
    # Mega-backdoor
    ('Inputs.spouse_*_mega_backdoor_enabled', 'bool',
     'enable after-tax 401(k) → in-plan Roth conversion'),
    ('Inputs.spouse_*_after_tax_401k_pct',    '0.0 - 0.50',
     'fraction of salary into after-tax 401(k); §415(c) capped'),
    # HSA after 65
    ('(automatic at age 65)',                 '—',
     'HSA becomes a 4th retirement bucket — drains for non-medical spending at ordinary rates'),
], columns=['knob', 'type / range', 'effect'])

with pd.option_context('display.max_colwidth', 90):
    print(_tier_b.to_string(index=False))

print()
print('Available state regimes:', [r.name for r in (STATELESS, CA, NY, IL, MA)])
print('Available CMA presets:  ', sorted(CMA_PRESETS))

## Recap — what changed across v2 → v4

| Capability | Where to set it | Effect on output |
|---|---|---|
| **v2** Stochastic returns | `cfg.market = LognormalModel(...)` | adds `prob_success`, percentile / CVaR metrics |
| **v2** Mortality / widow's penalty | `cfg.mortality = Mortality(year_of_death_a=...)` | flips filing status to single after death; SS / pension scale per election |
| **v2** Asset location | `cfg.asset_location = AssetLocation(...)` | per-account equity/bond split — bonds in pretax, equities in Roth |
| **v2** Smile spending + lumps | `cfg.spending = SpendingProfile.retirement_smile(...)` | per-age multipliers + lump events + LTC shock |
| **v2** Tax regime swap | `cfg.tax_regime`, `cfg.regime_change_year_offset` + `regime_change_target` | per-year regime selection |
| **v2** Optimizer objective | `optimize_s3(cfg, inputs, objective='cvar')` | CVaR or `'p_success'` instead of point-estimate terminal NW |
| **v3 Tier A** RMD-first ordering, expanded Roth conversions, full bracket indexing | corrects long-known modeling bugs | terminal NW shifts by 5-15% in many scenarios |
| **v3 Tier B** State income tax | `cfg.state_regime = CA / NY / IL / MA / STATELESS` | state-level brackets, exclusions, HSA add-backs |
| **v3 Tier B** Bequest tax in objective | `cfg.heir_marginal_rate = 0.22` | discounts terminal pretax & HSA when projecting to heirs |
| **v3 Tier B** Direct + backdoor Roth IRA | `inputs.spouse_*_roth_ira_contrib`, `_backdoor_roth` | MAGI phase-out + pro-rata aware |
| **v3 Tier B** Mega-backdoor Roth | `inputs.spouse_*_mega_backdoor_enabled`, `_after_tax_401k_pct` | §415(c) capped, routes to Roth |
| **v3 Tier B** HSA-after-65 | (automatic) | HSA becomes a 4th retirement bucket post-65 |
| **v4** Equity-bond correlation | `LognormalModel(equity_bond_corr=...)` | bivariate normal sampling — no more independent draws |
| **v4** CAPE-conditioned returns | `LognormalModel(cape_today=33.0)` | scales `equity_mu` by `cape_long_run / cape_today` |
| **v4** Historical sequence replay | `cfg.market = HistoricalSequenceModel()` | one contiguous 30y slice per path (Bengen / FIRECalc style) |
| **v4** CMA presets | `cfg.market = lognormal_from_cma("vanguard_2025")` | pick a published forward-looking assumption |

The `tax_optimizer/` package is the single source of truth — every cell
in this notebook imports from it. Run `python -m tax_optimizer --help`
for the equivalent CLI invocation.
